In [86]:
pip install requests scrapy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests  
import pandas as pd              # for sending HTTP requests

from scrapy import Selector    # for parsing HTML content

Part 1: Navigating the structure of a webpage (30 minutes)
🎯 ACTION POINTS

Open the homepage of the English version of Wikipedia in your web browser and inspect the HTML structure of the page.

Identify the path in the DOM tree to the Wikipedia logo () and write it down.

On most browsers, once you have inspected the element, you can right-click on the HTML code and select “Copy” > “Copy Selector” to get a CSS Selectyor code you could use in your Python script.
Send the request to the Wikipedia homepage using the requests package and store the HTML content in a variable called html.

In [4]:
url = "https://en.wikipedia.org/"
html = requests.get(url).content # This sends a GET request to the URL and stores the HTML 


In [5]:
url = "https://en.wikipedia.org/"
headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}
html = requests.get(url, headers=headers).text

Use the Selector class from the scrapy package to parse the HTML content. Save it in a variable called sel.

In [6]:
sel = Selector(text=html)

In [91]:
sel.css("title::text").get()  # Extracts the text within the <title> tagcontent of the response.

'Wikipedia, the free encyclopedia'

This object stores the full HTML content of the Wikipedia homepage, at the time you sent the request.

Use the css() method from the Selector object to extract the Wikipedia logo from the HTML, using the path you identified in step 2. Save it in a variable called url_logo. Here is an example of how you can do this:

In [92]:
url_logo = sel.css("img.mw-logo-wordmark")
url_logo

[<Selector query="descendant-or-self::img[@class and contains(concat(' ', normalize-space(@class), ' '), ' mw-logo-wordmark ')]" data='<img class="mw-logo-wordmark" alt="Wi...'>]

Print all the attributes of the Wikipedia logo using the attrib attribute of the Selector object.

In [93]:
url_logo.attrib

{'class': 'mw-logo-wordmark',
 'alt': 'Wikipedia',
 'src': '/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg',
 'style': 'width: 8.75em; height: 1.375em;'}

In [94]:
logo = sel.css("img.mw-logo-icon")[0]
url_logo = 'https://en.wikipedia.org/' + logo.attrib['src']
print(url_logo)

https://en.wikipedia.org//static/images/icons/enwiki-25.svg


### Part 2: Understanding the HTML Structure (30 minutes)

In [111]:
featured_article = sel.css("div#mp-tfa")
print(featured_article)

[<Selector query="descendant-or-self::div[@id = 'mp-tfa']" data='<div id="mp-tfa" class="mp-contains-f...'>]


In [112]:
featured_text = featured_article.css("::text").getall()

In [ ]:
featured_text_clean= " ".join(t.strip()for t in featured_text if t.strip())
featured_text_clean

'Ice-filled summit crater of Mount Edziza Mount Edziza is a volcanic mountain in Cassiar Land District in northwestern British Columbia , Canada. It is located on the Big Raven Plateau of the Tahltan Highland , which extends along the western side of the Stikine Plateau . Mount Edziza has an elevation of 2,786 metres (9,140 feet), making it the highest point of the Mount Edziza volcanic complex and one of the highest volcanoes in Canada. However, it had an elevation of at least 3,396\xa0m (11,142\xa0ft) before its formerly cone-shaped summit was likely destroyed by a violent eruption in the geologic past; its current flat summit contains an ice-filled crater (pictured) 2 kilometres (1.2 miles) in diameter. Mount Edziza contains several lava domes , cinder cones and lava fields on its flanks, as well as an ice cap containing several outlet glaciers that extend to lower elevations. All sides of the mountain are drained by tributaries of Mess Creek and Kakiddi Creek , which are situated w

In [113]:
featured_article_link = featured_article.css("a")

What is the data structure of the output? Is it a list, a dictionary, a string, or something else?

In [114]:
type(featured_article_link)


scrapy.selector.unified.SelectorList

In [115]:
len(featured_article_link)

26

In [116]:
len(featured_article)

1

In [117]:
print(featured_article)

[<Selector query="descendant-or-self::div[@id = 'mp-tfa']" data='<div id="mp-tfa" class="mp-contains-f...'>]


In [118]:
type(featured_article_link)

scrapy.selector.unified.SelectorList

In [120]:
len(featured_article.css("::text").getall())

67

In [121]:
data =[]
for a in featured_article_link:
    href = a.attrib.get('href')
    title = a.attrib.get('title')

    if href and title:
        data.append({
            'title': title,
            'href': href
        })
        

In [122]:
df = pd.DataFrame(data)
df.head()

,title,href
0,Ice-filled summit crater of Mount Edziza,/wiki/File:Edziza042909--_113-16.jpg
1,Mount Edziza,/wiki/Mount_Edziza
2,Cassiar Land District,/wiki/Cassiar_Land_District
3,British Columbia,/wiki/British_Columbia
4,Big Raven Plateau,/wiki/Big_Raven_Plateau


# Website crawl

In [123]:
first_link = df[df["href"].str.startswith("/wiki/")]['href'].iloc[0]
first_link
                                        

'/wiki/File:Edziza042909--_113-16.jpg'

In [125]:
base_url = "https://en.wikipedia.org"

In [126]:
first_link_url = base_url+ first_link
first_link_url

'https://en.wikipedia.org/wiki/File:Edziza042909--_113-16.jpg'

In [127]:
html2 = requests.get(first_link_url, headers=headers).text
html2[:500]  # Display the first 500 characters of the HTML content

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref--excluded vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled v'

In [129]:
sel2 = Selector(text=html2)

In [130]:
h2_text = sel2.css('h2::text').getall()

In [131]:
h2_text_clean =[h.strip() for h in h2_text if h.strip()]
h2_text_clean

['Summary',
 'Licensing',
 'File history',
 'File usage',
 'Global file usage',
 'Metadata']

In [132]:
first_link = df[
    df['href'].str.startswith("/wiki/") &
    ~df["href"].str.contains("File:")
]['href'].iloc[0]
print(first_link)

/wiki/Mount_Edziza


In [133]:
base_url = "https://en.wikipedia.org"
headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

In [148]:
results =[]

In [150]:
for link in df['href'].unique():
    if link.startswith("/wiki/") and "File:" not in link:

        full_url = base_url + link
        response = requests.get(full_url, headers=headers)

        if response.status_code != 200:
            continue
        html = response.text
        sel = Selector(text=html)

        page_title = sel.css("title::text").get()

        h2_headers = sel.css("h2::text").getall()
        h2_headers = [h.strip() for h in h2_headers if h.strip()]

        for h in h2_headers:
            results.append({
                'url': full_url,
                'page_title': page_title,
                'link': link,
                'header': h

            })

In [151]:
featured_article_h2 = pd.DataFrame(results)
featured_article_h2.head()

,url,page_title,link,header
0,https://en.wikipedia.org/wiki/Mount_Edziza,Mount Edziza - Wikipedia,/wiki/Mount_Edziza,Contents
1,https://en.wikipedia.org/wiki/Mount_Edziza,Mount Edziza - Wikipedia,/wiki/Mount_Edziza,Name and etymology
2,https://en.wikipedia.org/wiki/Mount_Edziza,Mount Edziza - Wikipedia,/wiki/Mount_Edziza,Geography and geomorphology
3,https://en.wikipedia.org/wiki/Mount_Edziza,Mount Edziza - Wikipedia,/wiki/Mount_Edziza,Geology
4,https://en.wikipedia.org/wiki/Mount_Edziza,Mount Edziza - Wikipedia,/wiki/Mount_Edziza,Human history


In [152]:
featured_article_h2.shape

(181, 4)

In [155]:
featured_article_h2.to_csv("featured_article_h2.csv", index=False)